# OCR hóa đơn — `train_receipt_ocr.ipynb` (Colab-safe)

**Tránh timeout Colab:**
1. **Runtime → Change runtime type → T4 GPU**
2. Chạy **CELL 0 → 1 → 2 → 3** (mount + dataset)
3. Train **từng model 1 cell** (CELL 4a → 4b → 4c → 4d) — mỗi cell ~8-12 phút
4. Sau mỗi cell train, model **tự backup lên Drive** → bị ngắt vẫn không mất
5. Phiên mới: chạy lại CELL 0→2 → **CELL 3.5** (restore) → chỉ chạy cell còn thiếu

**Luồng đầy đủ:** 0 → 1 → 2 → 3 → **3.5** → 4a → 4b → 4c → 4d → 5 (eval) → 6 (backup) → 7 (demo)

---
**TEST NHANH VỚI ẢNH THẬT (không cần train):**
Chỉ cần chạy **CELL 0 → 1 → CELL-EASYOCR** → upload ảnh bất kỳ → ra kết quả ngay.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 0 — Cài thư viện + Mount Google Drive
# ══════════════════════════════════════════════════════════════

import subprocess, sys, zipfile, shutil
from pathlib import Path

# ══════════════════════════════════════════════════════════════
# CELL 0 — Cài thư viện + EasyOCR + Mount Google Drive
# ══════════════════════════════════════════════════════════════

import subprocess, sys, zipfile, shutil
from pathlib import Path

# Thu vien co ban
for pkg in ["torch", "numpy", "pandas", "matplotlib", "Pillow"]:
    try:
        __import__("PIL" if pkg == "Pillow" else pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# EasyOCR — quan trong cho nhan dang anh that (co ho tro tieng Viet)
try:
    import easyocr
    print("EasyOCR da co san")
except ImportError:
    print("Dang cai EasyOCR (~2 phut lan dau)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "easyocr"])
    import easyocr
    print("EasyOCR da cai xong")

# Font cho Colab (tranh loi Windows path)
import subprocess as sp
sp.run(["apt-get", "install", "-y", "-qq", "fonts-dejavu-core", "fonts-liberation"],
       stdout=sp.DEVNULL, stderr=sp.DEVNULL)

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/expense_receipt_ocr")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Backup model se luu tai: {DRIVE_DIR}")
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/expense_receipt_ocr")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Backup model se luu tai: {DRIVE_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — Tim thu muc ai_service tren Drive
# ══════════════════════════════════════════════════════════════

AI_ROOT = None
CANDIDATES = [
    Path("/content/drive/MyDrive/thesis/ai_service"),
    Path("/content/drive/MyDrive/ai_service"),
    Path("/content/ai_service"),
]
for p in CANDIDATES:
    if (p / "app" / "ocr_net.py").is_file():
        AI_ROOT = p
        break
if AI_ROOT is None:
    for cand in Path("/content/drive/MyDrive").rglob("ocr_net.py"):
        if cand.parent.name == "app":
            AI_ROOT = cand.parent.parent
            break

assert AI_ROOT is not None, "Khong tim thay thesis/ai_service tren Drive"

# QUAN TRONG: copy code ve /content/ — KHONG chay script truc tiep tu Drive
# (Drive disconnect giua chung -> loi 'Transport endpoint is not connected')
WORK_ROOT = Path("/content/ai_work")
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
shutil.copytree(AI_ROOT, WORK_ROOT, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "models", "data/receipt_ocr"))
print(f"Copy code -> {WORK_ROOT} (chay tu day, khong tu Drive)")

# Kiem tra file can co
REQUIRED_OCR = ["ocr_net.py", "ocr_infer.py", "ocr_charset.py", "ocr_eval.py", "receipt_parse.py", "receipt_layout.py"]
REQUIRED_CLS = ["classify_infer.py", "classify_net.py", "rules.py"]
print(f"AI_ROOT  = {AI_ROOT}  (Drive)")
print(f"WORK_ROOT= {WORK_ROOT}  (local Colab)\n")
for f in REQUIRED_OCR:
    ok = (WORK_ROOT / "app" / f).is_file()
    print(f"  app/{f}: {'OK' if ok else 'THIEU — upload tu may'}")
REQUIRED_DATA = ["gen_receipt_dataset.py", "train_receipt_models.py", "colab_ocr_helpers.py", "eval_receipt_ocr.py"]
print("  --- data/ ---")
for f in REQUIRED_DATA:
    ok = (WORK_ROOT / "data" / f).is_file()
    print(f"  data/{f}: {'OK' if ok else 'THIEU — upload tu may'}")
print("  --- (CELL 7 demo danh muc) ---")
for f in REQUIRED_CLS:
    ok = (WORK_ROOT / "app" / f).is_file()
    print(f"  app/{f}: {'OK' if ok else 'THIEU — bo qua CELL 7'}")

%cd {WORK_ROOT}

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2 — GPU + tham so + DUONG DAN (quan trong tren Colab)
# ══════════════════════════════════════════════════════════════

import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cpu":
    print("CANH BAO: Dang dung CPU. Vao Runtime > Change runtime type > GPU")

GEN_N    = 5000   # 5000 bill — cân bằng chất lượng / thời gian Colab
EPOCHS   = 45     # early-stop thường dừng ~30-40 ep
PATIENCE = 15
FIELDS   = "amount,merchant,date,line"

DATA_DIR   = Path("/content/receipt_ocr")
MODELS_DIR = Path("/content/receipt_models")
LOG_DIR    = MODELS_DIR / "ocr_logs"
DATASET_ZIP = DRIVE_DIR / "receipt_ocr.zip"

print(f"GEN_N={GEN_N}  EPOCHS={EPOCHS}")
print(f"DATA_DIR   = {DATA_DIR}")
print(f"MODELS_DIR = {MODELS_DIR}")
print(f"DRIVE      = {DRIVE_DIR}")
print(f"Dataset zip tren Drive: {DATASET_ZIP.name}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3 — Dataset: restore tu Drive HOAC sinh moi (1 lan)
# ══════════════════════════════════════════════════════════════

import pandas as pd
import shutil
import sys
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
from data.colab_ocr_helpers import restore_dataset_from_drive, save_dataset_to_drive

manifest = DATA_DIR / "manifest.csv"

# Thu restore tu zip tren Drive (nhanh ~2 phut)
if not manifest.is_file():
    try:
        restored = restore_dataset_from_drive(DRIVE_DIR, DATA_DIR)
        if restored:
            manifest = DATA_DIR / "manifest.csv"
    except OSError as e:
        print(f"Drive loi khi restore: {e}")
        print("-> Chay lai CELL 0 de mount Drive, hoac sinh dataset moi")

if manifest.is_file():
    n = len(pd.read_csv(manifest))
    print(f"Da co dataset: {n:,} bill")
    if n < GEN_N * 0.9:
        print(f"Dataset nho ({n}) — xoa va sinh lai...")
        shutil.rmtree(DATA_DIR, ignore_errors=True)
        manifest = None

if not manifest.is_file():
    print(f"Sinh {GEN_N} bill vao {DATA_DIR} (~5-8 phut)...")
    print("KHONG bam Ctrl+C — cho den khi thay 'OK manifests'")
    # Chay tu WORK_ROOT (/content/), KHONG tu Drive
    !python {WORK_ROOT}/data/gen_receipt_dataset.py --n {GEN_N} --out-dir {DATA_DIR}
    if (DATA_DIR / "manifest.csv").is_file():
        try:
            save_dataset_to_drive(DATA_DIR, DRIVE_DIR)
        except OSError as e:
            print(f"Khong nen duoc zip len Drive: {e} — chay lai CELL 0 roi CELL 3")
    else:
        raise RuntimeError("Sinh dataset THAT BAI — xem loi o tren, chay lai CELL 3")

df = pd.read_csv(DATA_DIR / "manifest.csv")
print(f"Tong: {len(df):,} bill")
df.head(3)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3.5 — Restore model tu Drive + xem trang thai (CHAY MOI PHIEN)
# ══════════════════════════════════════════════════════════════

import sys
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

from data.colab_ocr_helpers import (
    restore_models_from_drive,
    list_training_status,
    field_done_on_drive,
)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
restored = restore_models_from_drive(DRIVE_DIR, MODELS_DIR)
if restored:
    print(f"Restore tu Drive: {restored}")
else:
    print("Chua co model tren Drive — bat dau train tu dau")

print("\n=== TRANG THAI TRAIN ===")
for field, st in list_training_status(DRIVE_DIR, MODELS_DIR).items():
    skip = "SKIP" if field_done_on_drive(field, DRIVE_DIR) else "CAN TRAIN"
    print(f"  {field:10s}  {st:25s}  [{skip}]")
print("\nChi chay cell 4a-4d cua field con thieu (CHUA TRAIN)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4a — Train AMOUNT (~8 phut) + backup Drive ngay
# ══════════════════════════════════════════════════════════════

from data.colab_ocr_helpers import field_done_on_drive, backup_field_to_drive, backup_logs_to_drive

FIELD = "amount"
if field_done_on_drive(FIELD, DRIVE_DIR):
    print(f"[{FIELD}] Da co tren Drive — bo qua. Xoa file tren Drive neu muon train lai.")
else:
    !python data/train_receipt_models.py --epochs {EPOCHS} --patience {PATIENCE} --fields {FIELD} --data-dir {DATA_DIR} --models-dir {MODELS_DIR} --log-dir {LOG_DIR}
    backup_field_to_drive(FIELD, MODELS_DIR, DRIVE_DIR)
    backup_logs_to_drive(LOG_DIR, DRIVE_DIR)
    print(f"[{FIELD}] XONG + da backup Drive")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4b — Train MERCHANT (~10 phut) + backup Drive
# ══════════════════════════════════════════════════════════════

FIELD = "merchant"
if field_done_on_drive(FIELD, DRIVE_DIR):
    print(f"[{FIELD}] Da co tren Drive — bo qua.")
else:
    !python data/train_receipt_models.py --epochs {EPOCHS} --patience {PATIENCE} --fields {FIELD} --data-dir {DATA_DIR} --models-dir {MODELS_DIR} --log-dir {LOG_DIR}
    backup_field_to_drive(FIELD, MODELS_DIR, DRIVE_DIR)
    backup_logs_to_drive(LOG_DIR, DRIVE_DIR)
    print(f"[{FIELD}] XONG + da backup Drive")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4c — Train DATE (~10 phut) + backup Drive
# FIX: img_w=320 (label date ~16 ky tu, can T>=16 timestep CTC)
# Neu model date cu bi loi (acc=0%): xoa file tren Drive truoc khi chay
# ══════════════════════════════════════════════════════════════

FIELD = "date"
FORCE_RETRAIN_DATE = True  # dat False sau khi train date OK

if FORCE_RETRAIN_DATE:
    for suffix in ("_model.pt", "_meta.json"):
        p = DRIVE_DIR / f"ocr_date{suffix}"
        if p.is_file():
            p.unlink()
            print(f"Xoa model date cu: {p.name}")

if field_done_on_drive(FIELD, DRIVE_DIR) and not FORCE_RETRAIN_DATE:
    print(f"[{FIELD}] Da co tren Drive — bo qua.")
else:
    !python data/train_receipt_models.py --epochs {EPOCHS} --patience {PATIENCE} --fields {FIELD} --data-dir {DATA_DIR} --models-dir {MODELS_DIR} --log-dir {LOG_DIR}
    backup_field_to_drive(FIELD, MODELS_DIR, DRIVE_DIR)
    backup_logs_to_drive(LOG_DIR, DRIVE_DIR)
    print(f"[{FIELD}] XONG + da backup Drive")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4d — Train LINE (~12 phut) + backup Drive
# ══════════════════════════════════════════════════════════════

FIELD = "line"
if field_done_on_drive(FIELD, DRIVE_DIR):
    print(f"[{FIELD}] Da co tren Drive — bo qua.")
else:
    !python data/train_receipt_models.py --epochs {EPOCHS} --patience {PATIENCE} --fields {FIELD} --data-dir {DATA_DIR} --models-dir {MODELS_DIR} --log-dir {LOG_DIR}
    backup_field_to_drive(FIELD, MODELS_DIR, DRIVE_DIR)
    backup_logs_to_drive(LOG_DIR, DRIVE_DIR)
    print(f"[{FIELD}] XONG + da backup Drive")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5 — DANH GIA (chay sau khi du 4 model)
# ══════════════════════════════════════════════════════════════

import pandas as pd
from pathlib import Path
import sys
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

from data.eval_receipt_ocr import (
    plot_loss_curves,
    show_dataset_samples,
    show_field_crop_samples,
    show_pred_vs_gt_with_images,
    plot_char_errors,
    run_full_eval,
    eval_real_bill,
)

LOG_DIR.mkdir(parents=True, exist_ok=True)

# 1) Metrics CER/WER + bang so sanh 4 model
compare = run_full_eval(DATA_DIR, MODELS_DIR, LOG_DIR, device=DEVICE)
display(compare)

# 2) Bieu do loss train/val theo epoch (4 model)
plot_loss_curves(LOG_DIR, LOG_DIR)

# 3) Anh dataset mau (full bill + crop tung field)
show_dataset_samples(DATA_DIR, LOG_DIR, n=6)
show_field_crop_samples(DATA_DIR, LOG_DIR, n=4)

# 4) Pred vs nhan that + phan tich loi ky tu
for field in ["amount", "merchant", "date", "line"]:
    show_pred_vs_gt_with_images(DATA_DIR, LOG_DIR, field, n=5)
    plot_char_errors(LOG_DIR, field)

# 5) Xem log CSV (luu cho bao cao)
for name in ["compare_models.csv", "amount_epoch_log.csv"]:
    p = LOG_DIR / name
    if p.is_file():
        print(f"\n--- {name} ---")
        display(pd.read_csv(p).head(10))

print(f"\nTat ca log/plot -> {LOG_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6 — Backup tong hop len Drive (neu chua backup tu cell 4a-4d)
# ══════════════════════════════════════════════════════════════

import json

models_dir = MODELS_DIR
prefixes = ["ocr_amount", "ocr_merchant", "ocr_date", "ocr_line"]

print("=" * 55)
for p in prefixes:
    meta_path = models_dir / f"{p}_meta.json"
    pt_path   = models_dir / f"{p}_model.pt"
    if meta_path.is_file():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        shutil.copy2(pt_path,   DRIVE_DIR / f"{p}_model.pt")
        shutil.copy2(meta_path, DRIVE_DIR / f"{p}_meta.json")
        print(f"  {p:14s}  val_loss={meta.get('val_ctc_loss', '?'):>8}  "
              f"CER={meta.get('mean_cer', '?'):>6}  acc={meta.get('exact_acc', '?'):>6}  "
              f"train={meta.get('train_samples', '?')}  val={meta.get('val_samples', '?')}")
    else:
        print(f"  {p:14s}  THIEU FILE — chay lai CELL 4")

# Backup log danh gia (CSV + PNG)
logs_src = MODELS_DIR / "ocr_logs"
logs_dst = DRIVE_DIR / "ocr_logs"
if logs_src.is_dir():
    if logs_dst.exists():
        shutil.rmtree(logs_dst)
    shutil.copytree(logs_src, logs_dst)
    n_files = len(list(logs_dst.rglob("*")))
    print(f"\n  ocr_logs/     {n_files} file (loss, CER, plots)")
else:
    print("\n  ocr_logs/     THIEU — chay lai CELL 4 hoac 4b")

print("=" * 55)
print(f"\nDa backup -> {DRIVE_DIR}")
print("Tai ve may: Drive > expense_receipt_ocr > 8 file .pt/.json + ocr_logs/")
print("Dat vao: c:\\Nam4\\Doantotnghiep2\\ai_service\\models\\")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6 — Demo parse bill (OCR bat buoc, classify tu chon)
# ══════════════════════════════════════════════════════════════

import sys
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

from PIL import Image
from app.ocr_infer import load_receipt_ocr_bundles
from app.receipt_parse import parse_receipt_image

ocr = load_receipt_ocr_bundles(MODELS_DIR, device=DEVICE)
assert ocr.amount is not None, "Chua train OCR — chay lai CELL 4"

# Classify (danh muc): can app/classify_infer.py + models/classify_*.pt
cls = None
cls_model_dir = WORK_ROOT / "models"
if (WORK_ROOT / "app" / "classify_infer.py").is_file():
    from app.classify_infer import load_classify_bundle
    cls = load_classify_bundle(cls_model_dir, device=DEVICE)
    if cls is None:
        print("Canh bao: co classify_infer.py nhung thieu classify_model.pt trong models/")
else:
    print("Canh bao: thieu classify_infer.py — chi demo OCR 4 field, category = rule-based")

sample = DATA_DIR / "images" / "full" / "bill_00000.png"
if sample.is_file():
    result = parse_receipt_image(Image.open(sample), ocr, cls)
    print(f"\nAnh: {sample.name}")
    for k, v in result.to_dict().items():
        print(f"  {k}: {v}")
else:
    print("Chua co anh — chay lai CELL 3")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7 (tu chon) — Test upload anh bill that + luu bao cao
# ══════════════════════════════════════════════════════════════

from google.colab import files
from PIL import Image
from io import BytesIO
from data.eval_receipt_ocr import eval_real_bill

print("Chon anh hoa don de test:")
uploaded = files.upload()

if uploaded:
    name = list(uploaded.keys())[0]
    img = Image.open(BytesIO(uploaded[name]))
    result = parse_receipt_image(img, ocr, cls)
    print(f"\nFile: {name}")
    for k, v in result.to_dict().items():
        print(f"  {k}: {v}")

    # Luu anh ket qua vao ocr_logs (cho bao cao)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    tmp = LOG_DIR / f"real_{name}"
    img.save(tmp)
    eval_real_bill(tmp, MODELS_DIR, DATA_DIR, LOG_DIR, device=DEVICE)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL EasyOCR — TEST BILL THẬT (chạy độc lập, không cần train)
# Chạy sau CELL 0 + CELL 1 là đủ
# ══════════════════════════════════════════════════════════════

import sys
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

from google.colab import files
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import json

from app.ocr_real import parse_receipt_easyocr, easyocr_available, _get_reader
from app.receipt_parse import _classify_from_text

# Tải sẵn EasyOCR reader (lần đầu ~30s, sau đó cache)
print("Load EasyOCR reader (tiếng Việt + English)...")
_get_reader(gpu=True if __import__('torch').cuda.is_available() else False)
print("Sẵn sàng!")

print("\nChọn ảnh bill / biên lai chuyển khoản để test:")
uploaded = files.upload()

for fname, data in uploaded.items():
    img = Image.open(BytesIO(data)).convert("RGB")
    
    # Hien thi anh
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(img)
    axes[0].set_title(f"Input: {fname}", fontsize=10)
    axes[0].axis("off")
    
    # Chay OCR
    result = parse_receipt_easyocr(img, gpu=__import__('torch').cuda.is_available())
    
    # Classify danh muc
    cls_bundle = None
    try:
        if (WORK_ROOT / "app" / "classify_infer.py").is_file():
            from app.classify_infer import load_classify_bundle
            cls_bundle = load_classify_bundle(MODELS_DIR, device=DEVICE)
    except Exception:
        pass
    cat, cat_conf, tx_type = _classify_from_text(cls_bundle, result.merchant or "", result.description or "")
    
    # Hien thi ket qua
    output = {
        "amount_vnd": result.amount_vnd,
        "transaction_date": result.transaction_date,
        "merchant": result.merchant,
        "description": result.description,
        "category": cat,
        "type": tx_type,
        "is_bank_transfer": result.is_bank_transfer,
        "confidence": {
            "amount": round(result.conf_amount, 3),
            "date": round(result.conf_date, 3),
            "merchant": round(result.conf_merchant, 3),
        }
    }
    
    txt = "\n".join(f"  {k}: {v}" for k, v in output.items() if k != "confidence")
    txt += f"\n  confidence:\n" + "\n".join(f"    {k}: {v}" for k,v in output["confidence"].items())
    txt += f"\n\n--- All text detected ---\n"
    txt += "\n".join(f"  {line}" for line in result.all_lines[:15])
    
    axes[1].text(0.02, 0.98, txt, va="top", ha="left", fontsize=9,
                 family="monospace", transform=axes[1].transAxes)
    axes[1].set_title("Kết quả OCR", fontsize=10)
    axes[1].axis("off")
    plt.tight_layout()
    plt.savefig(WORK_ROOT / f"easyocr_result_{fname}.png", dpi=100, bbox_inches="tight")
    plt.show()
    
    print(f"\n=== {fname} ===")
    print(json.dumps({k: v for k, v in output.items() if k != "confidence"}, ensure_ascii=False, indent=2))